# Portfolio data loader architecture demo

This notebook demonstrates the full standardized data-loading architecture:
- `CSVLoader`
- `YFinanceLoader`
- standardized output via `get_data()`
- `Portfolio` consuming loaders without depending on the source
- explicit notebook-level error handling for Yahoo Finance

In [1]:
from pathlib import Path

from portafolios.core.portafolio import Portfolio
from portafolios.data import CSVLoader, YFinanceLoader

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / "data_clean_stock_data.csv"
YF_SNAPSHOT_PATH = PROJECT_ROOT / "data" / "yf_snapshot.csv"

print("Project root:", PROJECT_ROOT)
print("Local CSV exists:", CSV_PATH.exists())
print("Yahoo snapshot exists:", YF_SNAPSHOT_PATH.exists())

Project root: c:\Users\narro\Desktop\semestre\honores_actuaria
Local CSV exists: True
Yahoo snapshot exists: False


## CSV standardized output

In [2]:
csv_loader = CSVLoader(
    path=CSV_PATH,
    tickers=["AAPL", "MSFT", "AMZN"],
    start="2020-01-01",
    end="2020-06-30",
    freq="B",
)

csv_data = csv_loader.get_data()
csv_data.prices.head()

,AAPL,MSFT,AMZN
Date,,,
2020-01-02,72.776606,153.428246,94.900497
2020-01-03,72.771729,152.683705,94.309998
2020-01-06,72.621654,151.872308,95.184502
2020-01-07,72.849231,152.416407,95.694504
2020-01-08,73.706271,153.495104,95.550003


In [3]:
print("CSV returns shape:", csv_data.returns.shape)
print("CSV tickers:", csv_data.tickers)
print("CSV metadata:", csv_data.metadata)

CSV returns shape: (128, 3)
CSV tickers: ['AAPL', 'MSFT', 'AMZN']
CSV metadata: {'source': 'csv', 'path': 'c:\\Users\\narro\\Desktop\\semestre\\honores_actuaria\\data_clean_stock_data.csv', 'frequency': 'B', 'start': '2020-01-02 00:00:00', 'end': '2020-06-30 00:00:00', 'n_assets': 3, 'n_observations': 129}


## Portfolio from CSVLoader

In [4]:
portfolio_csv = Portfolio(loader=csv_loader)
portfolio_csv.preparar_datos()
portfolio_csv.prices.head()

,AAPL,MSFT,AMZN
Date,,,
2020-01-02,72.776606,153.428246,94.900497
2020-01-03,72.771729,152.683705,94.309998
2020-01-06,72.621654,151.872308,95.184502
2020-01-07,72.849231,152.416407,95.694504
2020-01-08,73.706271,153.495104,95.550003


## Yahoo Finance with explicit error handling

This cell tries live download first. If it fails, it falls back to the saved snapshot if available.

In [5]:
portfolio_yf = None
yf_mode = None

try:
    yf_loader = YFinanceLoader(
        tickers=["AAPL", "MSFT", "AMZN"],
        start="2024-01-01",
        end="2024-03-31",
        batch_size=25,
        max_retries=4,
        retry_wait=3.0,
        timeout=30,
        progress=False,
        save_download=True,
        save_path=YF_SNAPSHOT_PATH,
        fallback_to_saved_data=False,
    )

    portfolio_yf = Portfolio(loader=yf_loader)
    portfolio_yf.preparar_datos()
    yf_mode = "live_download"
    print("Yahoo live download succeeded.")

except Exception as exc:
    print("Yahoo live download failed:")
    print(type(exc).__name__, str(exc))

    if not YF_SNAPSHOT_PATH.exists():
        raise RuntimeError(
            "Yahoo failed and no saved snapshot exists yet. "
            "Run this notebook later with internet access to create the snapshot."
        ) from exc

    print("Falling back to saved Yahoo snapshot...")
    yf_loader = YFinanceLoader(
        tickers=["AAPL", "MSFT", "AMZN"],
        start="2024-01-01",
        end="2024-03-31",
        save_path=YF_SNAPSHOT_PATH,
        use_saved_data=True,
    )
    portfolio_yf = Portfolio(loader=yf_loader)
    portfolio_yf.preparar_datos()
    yf_mode = "saved_snapshot"

print("Yahoo mode used:", yf_mode)
portfolio_yf.prices.head()

Yahoo live download succeeded.
Yahoo mode used: live_download


,AAPL,MSFT,AMZN
Date,,,
2024-01-02,183.731308,364.589417,149.929993
2024-01-03,182.355606,364.324036,148.470001
2024-01-04,180.039658,361.709106,144.570007
2024-01-05,179.317154,361.522278,145.240005
2024-01-08,183.652145,368.344757,149.100006


In [6]:
print("Yahoo metadata:", portfolio_yf.info)
print("Yahoo prices shape:", portfolio_yf.prices.shape)
print("Snapshot exists now:", YF_SNAPSHOT_PATH.exists())

Yahoo metadata: {'source': 'yfinance', 'frequency': '1d', 'start': '2024-01-02 00:00:00', 'end': '2024-03-28 00:00:00', 'n_assets': 3, 'n_observations': 61, 'save_path': 'c:\\Users\\narro\\Desktop\\semestre\\honores_actuaria\\data\\yf_snapshot.csv', 'used_saved_data': False}
Yahoo prices shape: (61, 3)
Snapshot exists now: True


## Saved-only Yahoo workflow

In [7]:
yf_saved_loader = YFinanceLoader(
    tickers=["AAPL", "MSFT", "AMZN"],
    start="2024-01-01",
    end="2024-03-31",
    save_path=YF_SNAPSHOT_PATH,
    use_saved_data=True,
)

portfolio_yf_saved = Portfolio(loader=yf_saved_loader)
portfolio_yf_saved.preparar_datos()
portfolio_yf_saved.prices.head()

,AAPL,MSFT,AMZN
Date,,,
2024-01-02,183.731308,364.589417,149.929993
2024-01-03,182.355606,364.324036,148.470001
2024-01-04,180.039658,361.709106,144.570007
2024-01-05,179.317154,361.522278,145.240005
2024-01-08,183.652145,368.344757,149.100006


In [8]:
print("CSV portfolio shape:", portfolio_csv.prices.shape)
print("Yahoo portfolio shape:", portfolio_yf.prices.shape)
print("Yahoo saved portfolio shape:", portfolio_yf_saved.prices.shape)

CSV portfolio shape: (129, 3)
Yahoo portfolio shape: (61, 3)
Yahoo saved portfolio shape: (61, 3)
